# Module 3: Operations Sanity Checks

## Overview

This notebook performs read-only validation and signal checks for your LangSmith production deployment. It assumes Module 1 and Module 2 are complete.

**⚠️ SAFETY NOTICE:** This notebook is **READ-ONLY**. It performs validation checks only and does NOT modify any infrastructure, Helm values, secrets, deployments, or resources. All operations are safe to run against production environments.

**Prerequisites:**
- Module 1 deployment is healthy and accessible
- Module 2 authentication is configured
- kubectl access to the cluster
- Read access to cloud provider APIs (for managed services)

## What We'll Check

1. ✅ Configuration (environment variables, redacted)
2. ✅ Preflight (kubectl context, namespace, deployments)
3. ✅ Current state snapshot (pods, services, events)
4. ✅ Early warning signals (restarts, pending pods, resource saturation)
5. ✅ Storage/durability checks (blob storage, backups)
6. ✅ Sidecar checks (Istio, if applicable)

**Estimated time:** 15-20 minutes

**Important:** 
- This notebook is read-only and safe to run. It does not modify any resources.
- All operations are read-only: `kubectl get`, `kubectl logs`, `kubectl top`, `helm get values`
- Artifacts are saved locally only (no cluster modifications)


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
possible_paths = [
    Path.cwd().parent,  # If cwd is module-3, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## Safety Check: Verify Environment

Before proceeding with validation, confirm you're working with the correct environment. This notebook is read-only and safe for production use.


In [ ]:
# Safety check: Verify environment and confirm read-only operations
from shared._cloud_helpers import get_cloud_provider, get_region, get_identity
from shared._validation import ok, warn

provider = get_cloud_provider()
region = get_region()
identity = get_identity()

print("### Environment Safety Check\n")

# Show current environment
provider_display = provider.upper()
print(f"Cloud Provider: {provider_display}")
print(f"Region: {region}")

if provider == "aws":
    print(f"Account ID: {identity.get('Account', 'N/A')}")
    print(f"User ARN: {identity.get('Arn', 'N/A')}")
elif provider == "azure":
    print(f"Subscription ID: {identity.get('SubscriptionId', identity.get('Account', 'N/A'))}")
    print(f"Subscription Name: {identity.get('SubscriptionName', 'N/A')}")

print("\n" + "=" * 60)
print("⚠️  IMPORTANT: This notebook is READ-ONLY")
print("=" * 60)
print("\nThis notebook will:")
print("  ✅ Validate production readiness")
print("  ✅ Check deployment status and health")
print("  ✅ Inspect resource usage and signals")
print("  ✅ Verify storage and backup configuration")
print("  ✅ Collect state snapshots (saved locally)")
print("\nThis notebook will NOT:")
print("  ❌ Modify Helm values or releases")
print("  ❌ Create or update secrets")
print("  ❌ Restart pods or deployments")
print("  ❌ Change any infrastructure")
print("  ❌ Modify any cluster resources")
print("\nAll operations are read-only:")
print("  - kubectl get (read resources)")
print("  - kubectl logs (read logs)")
print("  - kubectl top (read metrics)")
print("  - helm get values (read configuration)")
print("  - Write artifacts to local directory only")
print("\n" + "=" * 60)

ok("Environment safety check complete")
print("\n✅ Safe to proceed with read-only validation")


## 1. Configuration

Load and validate configuration from environment variables. All secrets are redacted in output.


In [ ]:
import os
import json
from shared._validation import require_env, print_config, redact, ok, warn
from shared._cloud_helpers import get_cloud_provider, get_region, get_identity

# Required configuration variables
required_vars = [
    "NAMESPACE",
    "CLUSTER_NAME",
]

# Optional but recommended
optional_vars = [
    "HELM_RELEASE",
    "LANGSMITH_DOMAIN",
]

print("### Loading Configuration\n")

# Load required variables
config = {}
missing = []

for var in required_vars:
    value = os.environ.get(var, "").strip()
    if not value:
        missing.append(var)
    config[var] = value

if missing:
    raise RuntimeError(f"❌ Missing required environment variables: {', '.join(missing)}\n"
                      f"💡 Copy env-samples/workshop.env.example to your .env file and fill in values")

# Load optional variables
for var in optional_vars:
    config[var] = os.environ.get(var, "").strip()

# Set defaults
if not config.get("HELM_RELEASE"):
    config["HELM_RELEASE"] = "langsmith"

# Print configuration (redacted)
print_config(config, redact_keys=set())

# Show cloud provider info
provider = get_cloud_provider()
region = get_region()
identity = get_identity()

provider_display = provider.upper()
print(f"\n### Current {provider_display} Session")
print(f"Cloud Provider: {provider_display}")
print(f"Region: {region}")

if provider == "aws":
    print(f"Account ID: {identity.get('Account', 'N/A')}")
elif provider == "azure":
    subscription_id = identity.get("SubscriptionId") or identity.get("Account", "")
    print(f"Subscription ID: {subscription_id}")

ok("Configuration loaded")


## 2. Preflight Checks

Verify kubectl context, namespace exists, and deployments are ready.


In [ ]:
from shared._validation import ok, warn
from shared._k8s_helpers import require_namespace, namespace_exists, wait_for_deployments_ready
from shared._shell import run

namespace = config["NAMESPACE"]
helm_release = config["HELM_RELEASE"]

print("### Preflight Checks\n")

# Check kubectl is available
print("1. Checking kubectl...")
result = run(["kubectl", "version", "--client", "--short"], check=False, stream=False)
if result.returncode == 0:
    ok("kubectl is available")
    print(f"   {result.stdout.strip()}")
else:
    raise RuntimeError("❌ kubectl is not available or not working")

# Check kubectl context
print("\n2. Checking kubectl context...")
result = run(["kubectl", "config", "current-context"], check=False, stream=False)
if result.returncode == 0:
    context = result.stdout.strip()
    ok(f"Current context: {context}")
else:
    warn("Could not determine kubectl context")
    print("   💡 Run: kubectl config get-contexts")

# Check namespace exists
print(f"\n3. Checking namespace '{namespace}'...")
if namespace_exists(namespace):
    ok(f"Namespace '{namespace}' exists")
else:
    raise RuntimeError(f"❌ Namespace '{namespace}' does not exist. Complete Module 1 first.")

# Check deployments are ready
print(f"\n4. Checking deployments...")
require_namespace(namespace)

try:
    wait_for_deployments_ready(namespace, timeout="2m")
    ok("All deployments ready")
except Exception as e:
    warn(f"Some deployments may not be ready: {e}")
    print("   💡 Check pod status manually: kubectl get pods -n {namespace}")

ok("Preflight checks complete")


## 3. Snapshot Current State

Capture current cluster state for baseline reference.


In [ ]:
from datetime import datetime
from shared._k8s_helpers import get_pods
from shared._shell import run

print("### Snapshotting Current State\n")

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
snapshot_dir = artifacts_dir / f"ops-snapshot-{timestamp}"
snapshot_dir.mkdir(exist_ok=True)

print(f"Saving snapshot to: {snapshot_dir}\n")

# 1. Get all resources
print("1. Capturing all resources...")
result = run(
    ["kubectl", "get", "all", "-n", namespace, "-o", "wide"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(snapshot_dir / "all-resources.txt", "w") as f:
        f.write(result.stdout)
    ok("All resources captured")
    print(result.stdout)
else:
    warn("Could not capture all resources")

# 2. Get events (sorted by timestamp)
print("\n2. Capturing recent events...")
result = run(
    ["kubectl", "get", "events", "-n", namespace, "--sort-by=.lastTimestamp"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(snapshot_dir / "events.txt", "w") as f:
        f.write(result.stdout)
    ok("Events captured")
    
    # Show recent events
    lines = result.stdout.strip().split("\n")
    if len(lines) > 1:
        print(f"\n   Last 10 events:")
        for line in lines[-10:]:
            print(f"   {line}")
else:
    warn("Could not capture events")

# 3. Get node and pod resource usage (if metrics available)
print("\n3. Checking resource usage...")
result = run(
    ["kubectl", "top", "nodes"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(snapshot_dir / "node-usage.txt", "w") as f:
        f.write(result.stdout)
    ok("Node usage captured")
    print(result.stdout)
else:
    warn("Node metrics not available (metrics-server may not be installed)")

result = run(
    ["kubectl", "top", "pods", "-n", namespace],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(snapshot_dir / "pod-usage.txt", "w") as f:
        f.write(result.stdout)
    ok("Pod usage captured")
    print(result.stdout)
else:
    warn("Pod metrics not available")

# 4. Check for data store services
print("\n4. Checking data store services...")
data_stores = {
    "postgres": ["postgres", "postgresql", "database", "db"],
    "redis": ["redis", "cache"],
    "clickhouse": ["clickhouse", "ch"],
}

found_stores = []
for store_type, keywords in data_stores.items():
    result = run(
        ["kubectl", "get", "svc", "-n", namespace, "-o", "json"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        services = json.loads(result.stdout)
        for svc in services.get("items", []):
            name = svc.get("metadata", {}).get("name", "").lower()
            if any(keyword in name for keyword in keywords):
                found_stores.append((store_type, name))
                print(f"   ✅ Found {store_type} service: {name}")

if found_stores:
    ok(f"Found {len(found_stores)} data store service(s)")
else:
    warn("No in-cluster data stores found (may be using managed services)")

ok(f"State snapshot saved to: {snapshot_dir}")


In [ ]:
from shared._validation import ok, warn
from shared._shell import run

print("### Early Warning Signal Checks\n")

# 1. Check pod restarts
print("1. Checking pod restart counts...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

critical_restarts = []
warning_restarts = []

if result.returncode == 0:
    pods = json.loads(result.stdout)
    for pod in pods.get("items", []):
        name = pod.get("metadata", {}).get("name", "")
        status = pod.get("status", {})
        phase = status.get("phase", "")
        
        # Check restart count
        container_statuses = status.get("containerStatuses", [])
        for cs in container_statuses:
            restart_count = cs.get("restartCount", 0)
            if restart_count > 5:
                critical_restarts.append((name, restart_count))
            elif restart_count > 2:
                warning_restarts.append((name, restart_count))
        
        # Check pod phase
        if phase == "CrashLoopBackOff":
            critical_restarts.append((name, "CrashLoopBackOff"))
        elif phase == "Pending":
            # Check how long it's been pending
            conditions = status.get("conditions", [])
            for cond in conditions:
                if cond.get("type") == "PodScheduled" and cond.get("status") != "True":
                    # Pod is pending
                    warning_restarts.append((name, "Pending"))

if critical_restarts:
    warn(f"❌ Critical: Found {len(critical_restarts)} pod(s) with critical issues")
    for pod_name, issue in critical_restarts:
        print(f"   - {pod_name}: {issue}")
    print("\n   💡 Action required: Check pod logs and events")
elif warning_restarts:
    warn(f"Found {len(warning_restarts)} pod(s) with warnings")
    for pod_name, issue in warning_restarts:
        print(f"   - {pod_name}: {issue}")
    print("\n   💡 Monitor these pods closely")
else:
    ok("No critical pod restart issues found")

# 2. Check for pending pods
print("\n2. Checking for pending pods...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "--field-selector=status.phase=Pending", "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pods = json.loads(result.stdout)
    pending = pods.get("items", [])
    if pending:
        warn(f"Found {len(pending)} pending pod(s)")
        for pod in pending:
            name = pod.get("metadata", {}).get("name", "")
            print(f"   - {name}")
        print("\n   💡 Check events: kubectl describe pod <name> -n {namespace}")
    else:
        ok("No pending pods")

# 3. Check resource saturation (if metrics available)
print("\n3. Checking resource saturation...")
result = run(
    ["kubectl", "top", "pods", "-n", namespace],
    check=False,
    stream=False
)

if result.returncode == 0:
    lines = result.stdout.strip().split("\n")[1:]  # Skip header
    saturated_pods = []
    
    for line in lines:
        parts = line.split()
        if len(parts) >= 3:
            pod_name = parts[0]
            cpu = parts[1]
            memory = parts[2]
            
            # Parse CPU (handle "m" suffix for millicores)
            try:
                if cpu.endswith("m"):
                    cpu_val = int(cpu[:-1])
                else:
                    cpu_val = int(float(cpu.replace("Gi", "").replace("Mi", "")))
                
                # Parse memory (handle "Mi" or "Gi" suffix)
                if "Gi" in memory:
                    mem_val = float(memory.replace("Gi", "")) * 1024
                elif "Mi" in memory:
                    mem_val = float(memory.replace("Mi", ""))
                else:
                    mem_val = 0
                
                # Check thresholds (simplified - would need requests/limits for accurate %)
                # For now, just flag very high absolute values
                if cpu_val > 2000:  # > 2 cores
                    saturated_pods.append((pod_name, f"High CPU: {cpu}"))
                if mem_val > 4096:  # > 4 Gi
                    saturated_pods.append((pod_name, f"High Memory: {memory}"))
            except (ValueError, IndexError):
                pass
    
    if saturated_pods:
        warn(f"Found {len(saturated_pods)} pod(s) with high resource usage")
        for pod_name, issue in saturated_pods:
            print(f"   - {pod_name}: {issue}")
        print("\n   💡 Review resource requests/limits and consider scaling")
    else:
        ok("No obvious resource saturation detected")
else:
    warn("Resource metrics not available (cannot check saturation)")

# 4. Check logs for common failure patterns
print("\n4. Checking logs for common failure patterns...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[*].metadata.name}"],
    check=False,
    stream=False
)

failure_patterns = {
    "connection refused": [],
    "timeout": [],
    "out of memory": [],
    "database error": [],
}

if result.returncode == 0 and result.stdout.strip():
    pod_names = result.stdout.strip().split()
    # Check API and worker pods (most likely to have issues)
    api_pods = [p for p in pod_names if any(keyword in p.lower() for keyword in ["api", "server", "backend"])]
    worker_pods = [p for p in pod_names if any(keyword in p.lower() for keyword in ["worker", "processor"])]
    
    pods_to_check = (api_pods[:2] if api_pods else []) + (worker_pods[:2] if worker_pods else [])
    
    for pod_name in pods_to_check[:4]:  # Check up to 4 pods
        try:
            log_result = run(
                ["kubectl", "logs", pod_name, "-n", namespace, "--tail=50"],
                check=False,
                stream=False
            )
            
            if log_result.returncode == 0:
                logs_lower = log_result.stdout.lower()
                
                for pattern, matches in failure_patterns.items():
                    if pattern in logs_lower:
                        # Check if it's actually an error (not just a log message)
                        lines = log_result.stdout.split("\n")
                        error_lines = [line for line in lines 
                                     if pattern in line.lower() 
                                     and any(err in line.lower() for err in ["error", "fail", "refused", "timeout"])]
                        
                        if error_lines:
                            matches.append((pod_name, len(error_lines)))
        except Exception:
            pass
    
    found_issues = False
    for pattern, matches in failure_patterns.items():
        if matches:
            found_issues = True
            warn(f"Found '{pattern}' pattern in {len(matches)} pod(s)")
            for pod_name, count in matches:
                print(f"   - {pod_name}: {count} occurrence(s)")
    
    if not found_issues:
        ok("No common failure patterns found in recent logs")
    else:
        print("\n   💡 Review pod logs for details: kubectl logs <pod> -n {namespace} --tail=100")
else:
    warn("Could not retrieve pod names for log checking")

ok("Early warning signal checks complete")


In [ ]:
from shared._validation import ok, warn
from shared._shell import run

print("### Storage / Durability Checks\n")

# 1. Check blob storage configuration
print("1. Checking blob storage configuration...")
result = run(
    ["kubectl", "get", "deployments", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

blob_storage_configured = False
blob_storage_provider = None

if result.returncode == 0:
    deployments = json.loads(result.stdout)
    for deployment in deployments.get("items", []):
        containers = deployment.get("spec", {}).get("template", {}).get("spec", {}).get("containers", [])
        
        for container in containers:
            env_vars = container.get("env", [])
            for env in env_vars:
                env_name = env.get("name", "").upper()
                env_value = env.get("value", "")
                
                # Check for blob storage configuration
                if "BLOB" in env_name or "S3" in env_name or "STORAGE" in env_name:
                    if "PROVIDER" in env_name:
                        blob_storage_provider = env_value
                        blob_storage_configured = True
                    elif env_value and env_value not in ["local", "filesystem", ""]:
                        blob_storage_configured = True

# Also check Helm values if accessible
helm_release = config.get("HELM_RELEASE", "langsmith")
result = run(
    ["helm", "get", "values", helm_release, "-n", namespace, "--output", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    try:
        values = json.loads(result.stdout)
        values_str = str(values).lower()
        
        # Look for blob storage configuration
        if "blob" in values_str or "s3" in values_str:
            if "local" not in values_str and "filesystem" not in values_str:
                blob_storage_configured = True
                if "s3" in values_str:
                    blob_storage_provider = "s3"
                elif "azure" in values_str:
                    blob_storage_provider = "azure"
    except json.JSONDecodeError:
        pass

if blob_storage_configured:
    if blob_storage_provider:
        ok(f"Blob storage configured: {blob_storage_provider}")
    else:
        ok("Blob storage appears configured (provider not detected)")
    print("   💡 Verify blob storage is NOT using local filesystem in production")
else:
    warn("❌ CRITICAL: Blob storage may not be configured")
    print("   💡 Blob storage is REQUIRED for production")
    print("   💡 Without it, ClickHouse will become unusable under load")
    print("   💡 Configure S3 (AWS) or Azure Blob Storage (Azure)")
    print("   💡 Check Helm values: helm get values <release> -n <namespace>")

# 2. Check for backup configuration indicators
print("\n2. Checking backup configuration...")
print("   Note: Backup configuration verification depends on deployment type")

# For managed services, we can't verify from cluster
# For in-cluster services, we can check for backup jobs
result = run(
    ["kubectl", "get", "cronjobs,jobs", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

backup_jobs_found = False
if result.returncode == 0:
    resources = json.loads(result.stdout)
    for item in resources.get("items", []):
        name = item.get("metadata", {}).get("name", "").lower()
        if "backup" in name:
            backup_jobs_found = True
            print(f"   ✅ Found backup job: {name}")

if backup_jobs_found:
    ok("Backup jobs found in cluster")
else:
    warn("No backup jobs found in cluster")
    print("   💡 For managed services (RDS, Azure Database), backups are automated")
    print("   💡 Verify backups in cloud provider console:")
    if provider == "aws":
        print("      - AWS RDS: Check automated backups in RDS console")
        print("      - AWS ElastiCache: Check snapshot configuration")
    elif provider == "azure":
        print("      - Azure Database: Check backup configuration in Azure portal")
        print("      - Azure Cache: Check backup configuration")
    print("   💡 For in-cluster ClickHouse, configure backup CronJob")

# 3. Check PVCs (for in-cluster storage)
print("\n3. Checking persistent volume claims...")
result = run(
    ["kubectl", "get", "pvc", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pvcs = json.loads(result.stdout)
    pvc_items = pvcs.get("items", [])
    
    if pvc_items:
        ok(f"Found {len(pvc_items)} PVC(s)")
        for pvc in pvc_items:
            name = pvc.get("metadata", {}).get("name", "")
            status = pvc.get("status", {}).get("phase", "")
            size = pvc.get("spec", {}).get("resources", {}).get("requests", {}).get("storage", "N/A")
            print(f"   - {name}: {status}, {size}")
        
        # Check for unbound PVCs
        unbound = [pvc for pvc in pvc_items if pvc.get("status", {}).get("phase") != "Bound"]
        if unbound:
            warn(f"Found {len(unbound)} unbound PVC(s)")
            print("   💡 Check storage class and node capacity")
    else:
        print("   No PVCs found (may be using managed services or ephemeral storage)")

ok("Storage / durability checks complete")


## 6. Sidecar Checks (Istio)

Detect if Istio sidecars are present and provide guidance on log access.


In [ ]:
from shared._validation import ok, warn
from shared._shell import run

print("### Sidecar Checks (Istio)\n")

# Check if Istio is installed (check for istiod)
result = run(
    ["kubectl", "get", "deployment", "-A", "-o", "jsonpath={.items[?(@.metadata.name==\"istiod\")].metadata.name}"],
    check=False,
    stream=False
)

istio_installed = False
if result.returncode == 0 and result.stdout.strip():
    istio_installed = True
    ok("Istio appears to be installed")
else:
    print("   Istio not detected (or not in default namespace)")
    print("   💡 Sidecar checks will be skipped")

if istio_installed:
    # Check for sidecar injection in namespace
    result = run(
        ["kubectl", "get", "namespace", namespace, "-o", "json"],
        check=False,
        stream=False
    )
    
    namespace_injection = False
    if result.returncode == 0:
        ns = json.loads(result.stdout)
        labels = ns.get("metadata", {}).get("labels", {})
        if labels.get("istio-injection") == "enabled" or labels.get("istio-discovery") == "enabled":
            namespace_injection = True
            ok("Namespace-level sidecar injection enabled")
            print(f"   Labels: {labels}")
        else:
            print("   Namespace-level injection not enabled")
            print("   💡 Sidecars may be injected per-workload")
    
    # Check for sidecars in pods
    print("\n2. Checking for sidecars in pods...")
    result = run(
        ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
        check=False,
        stream=False
    )
    
    pods_with_sidecars = []
    pods_without_sidecars = []
    
    if result.returncode == 0:
        pods = json.loads(result.stdout)
        for pod in pods.get("items", []):
            name = pod.get("metadata", {}).get("name", "")
            containers = pod.get("spec", {}).get("containers", [])
            container_names = [c.get("name", "") for c in containers]
            
            if "istio-proxy" in container_names:
                pods_with_sidecars.append((name, container_names))
            else:
                pods_without_sidecars.append((name, container_names))
    
    if pods_with_sidecars:
        ok(f"Found {len(pods_with_sidecars)} pod(s) with sidecars")
        print("\n   Pods with sidecars:")
        for pod_name, containers in pods_with_sidecars[:5]:  # Show first 5
            app_containers = [c for c in containers if c != "istio-proxy"]
            print(f"   - {pod_name}: {', '.join(app_containers)} + istio-proxy")
        
        print("\n   💡 Important: When fetching logs, specify container name:")
        print("      kubectl logs <pod> -n <namespace> -c <container-name>")
        print("      kubectl logs <pod> -n <namespace> -c istio-proxy  # for proxy logs")
        print("      kubectl logs <pod> -n <namespace> --all-containers=true  # for all logs")
        print("\n   ⚠️  If logs appear missing, you're likely looking at the wrong container!")
        
        if pods_without_sidecars:
            warn(f"Found {len(pods_without_sidecars)} pod(s) without sidecars")
            print("   💡 These pods may need sidecar injection or are opted out")
    else:
        if namespace_injection:
            warn("No pods with sidecars found (may need pod restart)")
            print("   💡 Existing pods require restart to get sidecars")
        else:
            print("   No sidecars detected (Istio may not be used or injection disabled)")

ok("Sidecar checks complete")


## Summary

### ✅ Sanity Checks Complete

This notebook has validated:
- ✅ Configuration loaded
- ✅ Preflight checks passed
- ✅ Current state snapshotted
- ✅ Early warning signals checked
- ✅ Storage/durability verified
- ✅ Sidecar status checked (if applicable)

### 🎯 Next Steps

1. **Review production readiness checklist:**
   - See `docs/shared/production_readiness_checklist.md`
   - Address any gaps identified

2. **Review signals and thresholds:**
   - See `docs/shared/ops_signals_and_thresholds.md`
   - Configure alerts based on thresholds

3. **Review sidecar documentation (if using Istio):**
   - See `docs/shared/sidecars_and_service_mesh.md`
   - Verify ServiceEntry configuration for external databases

4. **Document your baselines:**
   - Record current resource usage
   - Document scaling thresholds
   - Update runbooks with findings

### 📋 Common Issues Found

If checks failed, common issues include:
- Blob storage not configured (CRITICAL for production)
- Pods restarting (check logs and resource limits)
- Pending pods (check node capacity and PVC binding)
- High resource usage (review requests/limits)
- Missing backups (verify in cloud console)

### 🔍 Evidence for Support

When contacting support, include:
- State snapshot from this notebook
- Pod logs (from correct container if sidecars enabled)
- Recent events
- Resource usage metrics
- Configuration summary (redacted)

See `docs/shared/ops_signals_and_thresholds.md` for escalation evidence requirements.
